In [ ]:
# Parameters
city_key = "ywg"
baseline_feed_id = "2024-12-15"
comparison_feed_id = "current"
save_figures = False
figures_dir = "reports/pr2/figures"
dpi = 200

<cell_type>markdown</cell_type># PR2: Network Metrics and Transfer Burden

This notebook gives Cathy a fixed workflow: compare pre/post network structure, rank top hubs, and measure transfer burden among the busiest hubs.

## What you need to implement

All your work is in this notebook — no library code to write. The data loading and wiring cells are done; you just need to replace the `save_placeholder_figure` calls with your actual matplotlib plots. Required figures:

1. **`network_metrics_prepost.png`** — Diverging bar chart of network metric changes (Section 3)
2. **`transfer_heatmap.png`** — Heatmap of transfer hop counts between top hubs (Section 4)

Each figure cell has a story description of what to show and why. The data is already loaded into DataFrames in Section 2. Use `save_report_figure(fig, "filename.png", figures_dir=figure_output_directory, dpi=dpi, enabled=save_figures)` to export.

Key imports available: `plt`, `pd`, `sns`, `save_report_figure`, `save_placeholder_figure`. For PTN colors: `from ptn_analysis.analysis import PTN_TIER_COLORS`.

## 1. Setup & Helpers

In [ ]:
from pathlib import Path
import warnings

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import city2graph as c2g

warnings.filterwarnings("ignore")

from ptn_analysis import TransitContext
from ptn_analysis.analysis import (
    WEB_MERCATOR,
    NEIGHBOURHOOD_STYLE,
    POINT_MARKER_STYLE,
    LABEL_STYLE,
    add_consistent_basemap,
    plot_neighbourhood_base,
)
from ptn_analysis.context.reporting import (
    ensure_report_dirs,
    save_placeholder_figure,
    save_report_figure,
)

ensure_report_dirs("pr2")
figure_output_directory = Path(figures_dir)
figure_output_directory.mkdir(parents=True, exist_ok=True)

## 2. Load Network Outputs

In [ ]:
from shapely.geometry import LineString

ctx = TransitContext.from_defaults(feed_id=comparison_feed_id)
na = ctx.network()

# build_network_export_tables returns these exact keys:
#   "network_metrics_prepost"        -> CSV: network_metrics_prepost.csv
#   "top_hubs_current"               -> CSV: top_hubs_current.csv
#   "hub_transfer_burden"            -> CSV: hub_transfer_burden.csv
#   "network_communities_current"    -> CSV: network_communities_current.csv
network_exports = na.build_network_export_tables(
    baseline_feed_id=baseline_feed_id,
    top_n=20,
)
network_comparison_table = network_exports["network_metrics_prepost"]
top_hub_table = network_exports["top_hubs_current"]
transfer_burden_table = network_exports["hub_transfer_burden"]
community_table = network_exports["network_communities_current"]

# --- Build node & edge GeoDataFrames for network overlays ---
stops_df = na.stops_df()
network_nodes_gdf = gpd.GeoDataFrame(
    stops_df, geometry=gpd.points_from_xy(stops_df.stop_lon, stops_df.stop_lat),
    crs='EPSG:4326',
).set_index('stop_id')

edges_raw = na.edges_df()
_stop_coords = dict(zip(stops_df['stop_id'], zip(stops_df['stop_lon'], stops_df['stop_lat'])))
edge_geoms = []
for _, erow in edges_raw.iterrows():
    c1 = _stop_coords.get(erow['from_stop_id'])
    c2 = _stop_coords.get(erow['to_stop_id'])
    if c1 and c2:
        edge_geoms.append(LineString([c1, c2]))
    else:
        edge_geoms.append(None)
edges_raw = edges_raw.copy()
edges_raw['geometry'] = edge_geoms
network_edges_gdf = gpd.GeoDataFrame(
    edges_raw.dropna(subset=['geometry']),
    geometry='geometry', crs='EPSG:4326',
)

network_nodes_web = network_nodes_gdf.to_crs(WEB_MERCATOR)
network_edges_web = network_edges_gdf.to_crs(WEB_MERCATOR)
neigh_gdf = ctx.working_db.neighbourhood_gdf().to_crs(WEB_MERCATOR)

def strip_direction(name):
    """Strip direction prefixes and truncate to 20 chars."""
    if not isinstance(name, str):
        return str(name)[:20]
    for pfx in ['Northbound ', 'Southbound ', 'Eastbound ', 'Westbound ']:
        if name.startswith(pfx):
            name = name[len(pfx):]
            break
    return name[:20]

print(f"Network comparison rows: {len(network_comparison_table)}")
print(f"Top hubs: {len(top_hub_table)}")
print(f"Transfer rows: {len(transfer_burden_table)}")
print(f"Network nodes: {len(network_nodes_web)}, edges: {len(network_edges_web)}")
display(network_comparison_table)

In [ ]:
# Data is loaded — tables available:
#   network_comparison_table  (metric_name, baseline_value, comparison_value, absolute_change, pct_change)
#   top_hub_table             (stop_id, stop_name, in_degree, out_degree, total_degree)
#   transfer_burden_table     (origin_stop_name, destination_stop_name, path_hop_count)
#   community_table           (stop_id, community_id)
print("Tables loaded. Implement figures below.")


## 3. Network Metrics Comparison

In [ ]:
# Figure 1: Network Metrics Comparison — 2x2 grid, each metric gets its own panel

display(network_comparison_table)

metrics_to_show = ['node_count', 'edge_count', 'avg_degree', 'density']
plot_df = network_comparison_table[network_comparison_table['metric_name'].isin(metrics_to_show)].copy()
plot_df['metric_label'] = plot_df['metric_name'].map({
    'node_count': 'Nodes', 'edge_count': 'Edges',
    'avg_degree': 'Avg Degree', 'density': 'Density',
})

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, (_, row) in enumerate(plot_df.iterrows()):
    ax = axes[i]
    bv = row['baseline_value'] or 0
    cv = row['comparison_value'] or 0
    x = np.arange(2)
    colors = ['#4C72B0', '#DD8452']
    bars = ax.bar(x, [bv, cv], color=colors, edgecolor='gray', linewidth=0.5, width=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(['Pre-PTN', 'Current'], fontsize=10)

    # format values — place INSIDE bars to avoid clipping
    for j, v in enumerate([bv, cv]):
        fmt = f'{v:,.0f}' if v >= 10 else (f'{v:.2f}' if v >= 0.01 else f'{v:.5f}')
        ax.text(j, v * 0.85, fmt, ha='center', va='top', fontsize=10, fontweight='bold', color='white')

    ax.set_title(row['metric_label'], fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    ax.set_axisbelow(True)
    # Add headroom so % change badge isn't clipped
    ax.set_ylim(0, max(bv, cv) * 1.18)

    # % change annotation
    if bv and bv > 0:
        pct = 100 * (cv - bv) / bv
        sign = '+' if pct >= 0 else ''
        color = '#1a9850' if pct >= 0 else '#d73027'
        ax.text(0.95, 0.95, f'{sign}{pct:.1f}%', transform=ax.transAxes,
                ha='right', va='top', fontsize=11, fontweight='bold', color=color,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=color, alpha=0.9))

fig.suptitle('Network Structure: Pre-PTN vs Current', fontsize=13, fontweight='bold')
plt.tight_layout()
save_report_figure(fig, "network_metrics_prepost.png",
                   figures_dir=figure_output_directory, dpi=dpi, enabled=save_figures)
plt.close(fig)

## 4. Transfer Burden Heatmap

In [ ]:
# Figure 2: Transfer Burden Heatmap — cleaned labels, reordered, diagonal masked

transfer_clean = transfer_burden_table.copy()
transfer_clean['origin_label'] = transfer_clean['origin_stop_name'].apply(strip_direction)
transfer_clean['dest_label'] = transfer_clean['destination_stop_name'].apply(strip_direction)

pivot = transfer_clean.pivot_table(
    index='origin_label',
    columns='dest_label',
    values='path_hop_count',
    aggfunc='first',
)

# Reorder by median hop count for visual grouping
row_order = pivot.median(axis=1).sort_values().index
pivot = pivot.loc[row_order, row_order]

# Mask diagonal (self-transfers = 0)
mask = np.eye(len(pivot), dtype=bool)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    pivot, ax=ax, cmap='YlOrRd', annot=True, fmt='.0f',
    mask=mask, square=True,
    linewidths=0.5, linecolor='white',
    annot_kws={'fontsize': 7},
    cbar_kws={'label': 'Transfer Hops', 'shrink': 0.6},
)
ax.set_title('Hub-to-Hub Transfer Burden (hop count)', fontsize=13, fontweight='bold')
ax.set_xlabel('Destination Hub', fontsize=11)
ax.set_ylabel('Origin Hub', fontsize=11)
ax.tick_params(axis='x', rotation=45, labelsize=8)
ax.tick_params(axis='y', rotation=0, labelsize=8)
plt.tight_layout()

save_report_figure(fig, "transfer_heatmap.png",
                   figures_dir=figure_output_directory, dpi=dpi, enabled=save_figures)
plt.close(fig)

## 5. Interpretation

Use the exported network comparison, top-hub, and transfer-burden tables directly in the report and dashboard. Community detection is retained for optional follow-up mapping, not as a blocker for the core network deliverables.

## 6. Hub Ranking (Top 20)

In [ ]:
# Hub Ranking — Top 20 by total degree (horizontal bar chart)

hub_plot = top_hub_table.head(20).copy()
hub_plot['label'] = hub_plot['stop_name'].apply(strip_direction)
hub_plot = hub_plot.sort_values('total_degree', ascending=True)

fig, ax = plt.subplots(figsize=(10, 10))
bars = ax.barh(range(len(hub_plot)), hub_plot['total_degree'],
               color='#4C72B0', edgecolor='gray', linewidth=0.3)
ax.set_yticks(range(len(hub_plot)))
ax.set_yticklabels(hub_plot['label'], fontsize=8)
for i, val in enumerate(hub_plot['total_degree']):
    ax.text(val + 0.3, i, f"{int(val)}", va='center', fontsize=8, fontweight='bold')
ax.set_xlabel('Total Degree (in + out)', fontsize=11)
ax.set_title('Top 20 Transit Hubs by Degree Centrality', fontsize=13, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
ax.set_axisbelow(True)
ax.set_ylim(-0.5, len(hub_plot) - 0.5)
plt.tight_layout()
save_report_figure(fig, "hub_ranking.png",
                   figures_dir=figure_output_directory, dpi=dpi, enabled=save_figures)
plt.close(fig)

if not community_table.empty:
    print(f"{community_table['community_id'].nunique()} Louvain communities detected")

## 7. Louvain Community Summary

## 7a. Weighted Centrality Comparison

In [ ]:
# Figure: Weighted vs Unweighted Hub Centrality Map
# Deduplicate spatially: keep highest-scoring stop per ~300m cluster
# Both panels use the FULL CITY extent (from neighbourhood boundaries) for consistency

try:
    centrality_comp = na.weighted_centrality_comparison(top_n=50)
    stops = na.stops_df()
    centrality_map = centrality_comp.merge(
        stops[['stop_id', 'stop_lat', 'stop_lon']], on='stop_id', how='left'
    ).dropna(subset=['stop_lat', 'stop_lon'])

    gdf_all = gpd.GeoDataFrame(
        centrality_map,
        geometry=gpd.points_from_xy(centrality_map['stop_lon'], centrality_map['stop_lat']),
        crs='EPSG:4326',
    ).to_crs(WEB_MERCATOR)

    def _spatial_dedup(gdf, col, n=15, min_dist_m=300):
        """Keep top-n stops by `col`, enforcing minimum spatial distance."""
        ranked = gdf.sort_values(col, ascending=False).copy()
        kept = []
        for _, row in ranked.iterrows():
            pt = row.geometry
            if any(pt.distance(prev.geometry) < min_dist_m for prev in kept):
                continue
            kept.append(row)
            if len(kept) >= n:
                break
        return gpd.GeoDataFrame(kept, crs=gdf.crs)

    def _scale(s):
        mn, mx = s.min(), s.max()
        if mx == mn:
            return np.full(len(s), 200.0)
        return 50 + 450 * (s - mn) / (mx - mn)

    # Use FULL CITY extent from neighbourhood boundaries (consistent with other maps)
    city_bounds = neigh_gdf.total_bounds
    xpad = (city_bounds[2] - city_bounds[0]) * 0.03
    ypad = (city_bounds[3] - city_bounds[1]) * 0.03

    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    offsets_cycle = [(12, 12), (-12, 16), (12, -16), (-14, -12), (16, 4),
                     (-16, 4), (12, -20), (-12, -20), (18, 0), (-18, 0)]

    for ax, col, title, cmap in [
        (axes[0], 'betweenness', 'Unweighted Betweenness', 'YlOrRd'),
        (axes[1], 'weighted_betweenness', 'Weighted Betweenness\n(travel-time)', 'YlGnBu'),
    ]:
        gdf = _spatial_dedup(gdf_all, col, n=15, min_dist_m=300)

        plot_neighbourhood_base(ax, neigh_gdf)
        network_edges_web.plot(ax=ax, color='#aaa', linewidth=0.15, alpha=0.3)
        gdf.plot(
            ax=ax, column=col, cmap=cmap, markersize=_scale(gdf[col]),
            alpha=0.85, edgecolor='gray', linewidth=0.2, legend=True,
            legend_kwds={'label': col.replace('_', ' ').title(), 'shrink': 0.5},
        )
        for idx, (_, row) in enumerate(gdf.nlargest(5, col).iterrows()):
            ox, oy = offsets_cycle[idx % len(offsets_cycle)]
            ax.annotate(
                strip_direction(row['stop_name']),
                xy=(row.geometry.x, row.geometry.y),
                xytext=(ox, oy), textcoords='offset points', fontsize=7,
                fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='gray', lw=0.6),
                bbox=dict(boxstyle='round,pad=0.15', facecolor='white', edgecolor='gray', alpha=0.85),
            )
        # Full city zoom — consistent with all other maps
        ax.set_xlim(city_bounds[0] - xpad, city_bounds[2] + xpad)
        ax.set_ylim(city_bounds[1] - ypad, city_bounds[3] + ypad)
        add_consistent_basemap(ax)
        ax.set_title(title, fontsize=12, fontweight='bold')

    fig.suptitle('Top 15 Hubs (spatially diverse): Unweighted vs Weighted Centrality',
                 fontsize=13, fontweight='bold', y=0.98)
    plt.tight_layout()
    save_report_figure(fig, "weighted_centrality_comparison.png",
                       figures_dir=figure_output_directory, dpi=dpi, enabled=save_figures)
    plt.close(fig)

except Exception as e:
    print(f"Could not generate weighted centrality figure: {e}")
    import traceback; traceback.print_exc()
    save_placeholder_figure('weighted_centrality_comparison.png', f'Error: {e}',
                            figures_dir=figure_output_directory, dpi=dpi, enabled=save_figures)

## 7b. Community Boundary Analysis

In [ ]:
# Figure: Louvain Community vs Neighbourhood Boundaries — edges colored by community
# Label each community with its dominant neighbourhood name

from matplotlib.patches import Patch

if stops_df.empty or community_table.empty:
    print("Missing stop or community data.")
    save_placeholder_figure('community_boundary_alignment.png', 'Missing data',
                            figures_dir=figure_output_directory, dpi=dpi, enabled=save_figures)
else:
    stops_with_comm = stops_df.merge(
        community_table[['stop_id', 'community_id']], on='stop_id', how='inner'
    )
    if stops_with_comm.empty:
        print("No community assignments found.")
        save_placeholder_figure('community_boundary_alignment.png', 'No community assignments',
                                figures_dir=figure_output_directory, dpi=dpi, enabled=save_figures)
    else:
        # Assign community to edges via from_stop_id
        comm_lookup = dict(zip(stops_with_comm['stop_id'], stops_with_comm['community_id']))
        edges_comm = network_edges_web.copy()
        edges_comm['community_id'] = edges_comm['from_stop_id'].map(comm_lookup)
        edges_comm = edges_comm.dropna(subset=['community_id'])
        edges_comm['community_id'] = edges_comm['community_id'].astype(int)

        # Map stops to neighbourhoods for labeling
        try:
            stop_nb = ctx.working_db.query("""
                SELECT s.stop_id, n.name AS neighbourhood
                FROM ywg_stops s
                JOIN ywg_neighbourhoods n ON ST_Contains(n.geometry, ST_Point(s.stop_lon, s.stop_lat))
                WHERE s.feed_id = :fid
            """, {"fid": comparison_feed_id})
            comm_nb = stops_with_comm.merge(stop_nb, on='stop_id', how='left')
            # Dominant neighbourhood per community
            dom_nb = (comm_nb.groupby('community_id')['neighbourhood']
                      .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else ''))
        except Exception:
            dom_nb = pd.Series(dtype=str)

        # Use 10 distinct colors that are easier to tell apart
        distinct_colors = ['#e6194b', '#3cb44b', '#4363d8', '#f58231', '#42d4f4',
                           '#f032e6', '#fabed4', '#469990', '#dcbeff', '#9A6324',
                           '#800000', '#aaffc3', '#000075', '#a9a9a9', '#ffe119']
        comm_sizes = edges_comm.groupby('community_id').size().sort_values(ascending=False)
        unique_comms = comm_sizes.index.tolist()
        comm_colors = {c: distinct_colors[i % len(distinct_colors)] for i, c in enumerate(unique_comms)}

        fig, ax = plt.subplots(figsize=(14, 10))
        plot_neighbourhood_base(ax, neigh_gdf, facecolor='none', edgecolor='#999', linewidth=0.4)

        for comm_id in unique_comms:
            subset = edges_comm[edges_comm['community_id'] == comm_id]
            subset.plot(ax=ax, color=comm_colors[comm_id], linewidth=0.4, alpha=0.7)

        # Label top 8 communities at their centroid with dominant neighbourhood name
        stops_web = gpd.GeoDataFrame(
            stops_with_comm,
            geometry=gpd.points_from_xy(stops_with_comm['stop_lon'], stops_with_comm['stop_lat']),
            crs='EPSG:4326',
        ).to_crs(WEB_MERCATOR)

        for comm_id in unique_comms[:8]:
            comm_stops = stops_web[stops_web['community_id'] == comm_id]
            if comm_stops.empty:
                continue
            cx_pt = comm_stops.geometry.x.mean()
            cy_pt = comm_stops.geometry.y.mean()
            label = dom_nb.get(comm_id, f'C{comm_id}')
            if not label:
                label = f'C{comm_id}'
            ax.annotate(
                label[:18], (cx_pt, cy_pt), fontsize=7, fontweight='bold',
                ha='center', va='center', color=comm_colors[comm_id],
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor=comm_colors[comm_id], alpha=0.85),
            )

        top_comms = unique_comms[:10]
        legend_patches = [Patch(color=comm_colors[c],
                                label=f'{dom_nb.get(c, f"C{c}")[:15]} ({comm_sizes[c]} edges)')
                          for c in top_comms]
        ax.legend(handles=legend_patches, loc='lower left', fontsize=7, framealpha=0.9)

        add_consistent_basemap(ax)
        ax.set_title('Louvain Communities and Neighbourhood Boundaries',
                     fontsize=13, fontweight='bold')
        plt.tight_layout()
        save_report_figure(fig, 'community_boundary_alignment.png',
                           figures_dir=figure_output_directory, dpi=dpi, enabled=save_figures)
        plt.close(fig)